# 02 — Pré-processamento

Cada decisão de limpeza, binarização e engenharia de atributos é justificada com base
na distribuição observada no notebook 01, não por convenção.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append(str(Path("..").resolve()))
from src.config import (
    RANDOM_STATE, DATA_RAW, DATA_PROCESSED, RESULTS, FIGURES, METRICS, MODELS,
    RAW_FILE, PROCESSED_FILE, ID_COL, TARGET, TARGET_BIN, QUALITY_THRESHOLD,
    TEST_SIZE, CV_FOLDS,
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

In [2]:
df = pd.read_csv(RAW_FILE)
df = df.drop(columns=[ID_COL])
df.shape

(1143, 12)

## 1. Dados faltantes

In [3]:
from src.preprocessing import check_missing

check_missing(df)

,nulos,pct


**Decisão:** o dataset não tem nenhum valor nulo (a tabela acima vem vazia, e isso
já havia sido confirmado em `df.info()` no notebook 01). Nenhuma imputação é
necessária.

## 2. Definição da variável alvo

In [4]:
for t in (6, 7, 8):
    pct = (df[TARGET] >= t).mean() * 100
    print(f"quality >= {t}: {pct:5.2f}% classificado como 'alta qualidade'")

quality >= 6: 54.33% classificado como 'alta qualidade'
quality >= 7: 13.91% classificado como 'alta qualidade'
quality >= 8:  1.40% classificado como 'alta qualidade'


**Decisão:** o limiar escolhido é **`quality >= 7`**, exatamente o sugerido no
enunciado do desafio. A tabela acima mostra por que ele é o mais adequado entre as
alternativas testadas: `quality >= 6` deixa 54,33% das amostras como "alta qualidade" —
não segmenta nada, é quase a base inteira; `quality >= 8` deixa só 1,40% — poucochíssimos
exemplos positivos para qualquer modelo aprender de forma estável. `quality >= 7` fica no
meio, com 13,91% de positivos: uma fatia pequena o bastante para representar um segmento
"premium" de fato, mas com exemplos suficientes (159 amostras) para treinar e validar
com alguma confiança.

In [5]:
from src.preprocessing import binarize_target

df = binarize_target(df, TARGET, QUALITY_THRESHOLD)
assert f"{TARGET}_bin" == TARGET_BIN, "nome da coluna binarizada não bate com TARGET_BIN"
df[TARGET_BIN].value_counts(normalize=True).round(4) * 100

quality_bin
0    86.09
1    13.91
Name: proportion, dtype: float64

## 3. Normalização / padronização

A padronização (`StandardScaler`) é aplicada dentro de um `Pipeline` no notebook 03,
não aqui — ajustar o scaler antes do split vazaria informação do conjunto de teste
para o treino.

**Decisão:** `StandardScaler` (média 0, desvio-padrão 1), não `MinMaxScaler`. O
motivo está na seção 4 do notebook 01: várias variáveis têm cauda longa e outliers
legítimos (`chlorides`, `residual sugar`, `sulphates`) — um único valor muito alto
comprimiria a normalização min-max de toda a coluna para o resto das amostras.
`StandardScaler` não depende de um mínimo/máximo fixo, então não sofre desse efeito.
A padronização só é usada pela Regressão Logística; Random Forest e Gradient Boosting,
por serem baseados em árvores de decisão, são invariantes à escala das variáveis e não
precisam dela.

O ajuste (`fit`) do scaler acontece **somente dentro do `Pipeline`, no notebook 03,
usando apenas o fold/partição de treino** — nunca aqui, com o dataset inteiro. Ajustar o
scaler antes do split faria estatísticas do conjunto de teste vazarem para o treino
(a média e o desvio usados para escalar já "veriam" dados que deveriam ser
desconhecidos), inflando artificialmente a performance medida.

## 4. Feature engineering

In [6]:
from src.preprocessing import build_features

df = build_features(df)
df[["acidity_ratio", "alcohol_sulphates"]].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
acidity_ratio,1143.0,18.088,9.266,4.808,11.613,15.102,22.042,74.167
alcohol_sulphates,1143.0,6.885,1.944,3.003,5.527,6.460,7.800,19.305


In [7]:
df[["acidity_ratio", "alcohol_sulphates", TARGET_BIN]].corr(numeric_only=True)[TARGET_BIN]

acidity_ratio        0.318700
alcohol_sulphates    0.352683
quality_bin          1.000000
Name: quality_bin, dtype: float64

**Decisão:** foram criadas duas variáveis derivadas, ambas com correlação superior à de
pelo menos uma das variáveis originais que as compõem:

- **`acidity_ratio`** (`fixed acidity / volatile acidity`): correlação de **0,32** com a
  variável resposta binária — superior à de `fixed acidity` isolada (0,12) e ligeiramente
  superior, em módulo, à de `volatile acidity` isolada (−0,31). A justificativa é
  enológica: a acidez fixa contribui para o frescor, enquanto a acidez volátil em excesso
  é percebida como defeito sensorial; a razão entre ambas expressa esse equilíbrio de
  forma mais completa do que cada termo isolado.
- **`alcohol_sulphates`** (`alcohol × sulphates`): correlação de **0,35** com a variável
  resposta — inferior à de `alcohol` isolado (0,40, a variável de maior correlação do
  conjunto), porém superior à de `sulphates` isolado (0,21). A finalidade é disponibilizar
  ao modelo linear o acesso direto à interação entre as duas variáveis de maior correlação
  positiva individual, informação que a Regressão Logística não captura sem um termo
  explícito. Modelos baseados em árvore representam interações de forma nativa, mas a
  inclusão do atributo não os prejudica e, conforme demonstrado no notebook 04, ele figura
  como a variável de maior importância nos três modelos treinados.

Nenhum outro atributo derivado foi mantido: a razão entre SO₂ livre e total, bem como
combinações de acidez total, foram testadas e descartadas por apresentarem correlação
inferior à das duas variáveis acima, sem ganho de interpretabilidade que justificasse a
ampliação do número de colunas.

## 5. Salvar dataset tratado

In [8]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_FILE, index=False)
df.shape

(1143, 15)

**Atenção — vazamento de dados:** a coluna `quality` (nota original, 3 a 8) é
mantida no arquivo salvo apenas para rastreabilidade/auditoria. Ela **não pode** entrar
como feature (`X`) na modelagem: `quality_bin` foi derivada diretamente dela, então
usá-la como preditora seria vazamento perfeito — o modelo "acertaria" 100% olhando para
a própria resposta disfarçada. O notebook 03 descarta explicitamente `quality` e
`quality_bin` de `X` antes de treinar qualquer modelo.